# Incident 4 — The Rogue Sub-Agent In A Fleet

**What happened:** A CrewAI research fleet shared a $1,000 budget. One sub-agent hallucinated a tool parameter and called a search API in a tight loop. It exhausted the **entire fleet budget**. The analyst and writer agents couldn't complete their tasks.

**Why it happens:** All agents share a single session token. A rogue agent consuming the shared pool starves every other agent — there is no per-agent ceiling.

**The fix:** Each sub-agent gets a scoped delegation token with its own hard cap. The researcher is capped at $200 — regardless of what it does, the analyst ($300) and writer ($200) allocations are untouched. The rogue agent harms only itself.

In [ ]:
import sys
!pip install "figuard>=0.3.2" --upgrade -q
for _mod in list(sys.modules.keys()):
    if _mod.startswith("figuard"):
        del sys.modules[_mod]
import figuard
print(f"✓ FiGuard {figuard.__version__} ready")

## Without FiGuard — rogue researcher drains the whole fleet

All agents share the same $1,000 pool. Once the researcher exhausts it, analyst and writer are blocked.

In [ ]:
# WITHOUT FiGuard — shared pool, no per-agent cap

fleet_budget = 1000.00
shared_balance = fleet_budget

def charge(agent, amount):
    global shared_balance
    if shared_balance >= amount:
        shared_balance -= amount
        return True
    return False

print(f"Fleet: ${fleet_budget:.2f}  |  researcher $200 cap  analyst $300  writer $200")
print()
print("Researcher goes rogue (tight search API loop)...")

researcher_spent = 0.0
for call in range(1, 1000):
    if not charge("researcher", 5.00):
        print(f"  Budget exhausted at call {call}")
        break
    researcher_spent += 5.00
    if call % 50 == 0 or call <= 2:
        print(f"  call {call:3d}: AUTHORIZED  (${researcher_spent:.2f} drained from fleet)")

print()
analyst_ok = charge("analyst", 50.00)
writer_ok = charge("writer", 40.00)
print(f"Analyst: {'AUTHORIZED' if analyst_ok else 'DENIED — fleet exhausted'}")
print(f"Writer:  {'AUTHORIZED' if writer_ok else 'DENIED — fleet exhausted'}")
print()
print(f"Researcher consumed: ${researcher_spent:.2f} of ${fleet_budget:.2f} fleet budget")
print(f"Fleet destroyed: analyst={not analyst_ok}, writer={not writer_ok}")

## With FiGuard — delegation tokens cap each agent independently

Each sub-agent holds a delegation token with its own spending cap. The researcher's loop stops at $200. The analyst and writer tokens are unaffected — they authorize against their own caps from the same fleet budget.

In [ ]:
from figuard import FiGuardClient

client = FiGuardClient(
    api_key="sb_live_demo",
    base_url="https://figuard-sandbox-1.onrender.com",
)

fleet = client.create_budget(
    user_id="crew_manager",
    total_limit=1000.00,
    currency="USD",
    expires_in="2h",
)

# Extract session token from tokens array
fleet_tokens = {t.category: t.session_token for t in fleet.tokens}
fleet_session_token = fleet_tokens["default"]


researcher_token = client.create_delegation_token(
    budget_id=fleet.id,
    session_token=fleet_session_token,
    label="researcher",
    caps=[{"category": "search_api", "limit": 200.00}],
    expires_in="2h",
)
analyst_token = client.create_delegation_token(
    budget_id=fleet.id,
    session_token=fleet_session_token,
    label="analyst",
    caps=[{"category": "llm_calls", "limit": 300.00}],
    expires_in="2h",
)
writer_token = client.create_delegation_token(
    budget_id=fleet.id,
    session_token=fleet_session_token,
    label="writer",
    caps=[{"category": "llm_calls", "limit": 200.00}],
    expires_in="2h",
)

print(f"Fleet: ${fleet.total_limit:.2f}  |  researcher $200  analyst $300  writer $200")
print()
print("Researcher goes rogue (tight search API loop)...")

spent = 0.0
for call in range(1, 1000):
    auth = client.authorize(
        session_token=researcher_token.session_token,
        agent_id="researcher",
        action_type="TOOL_CALL",
        description=f"Search API call {call}",
        requested_quantity=5.00,
        claimed_category="search_api",
        idempotency_key=f"search-{call}",
    )
    if not auth.is_authorized:
        print(f"✓ Researcher stopped at call {call}: {auth.denial_reason}")
        print(f"  Researcher spent: ${spent:.2f} of $200.00 cap")
        break
    spent += 5.00
    if call % 10 == 0 or call <= 2:
        print(f"  call {call:3d}: AUTHORIZED  (${spent:.2f} of $200.00)")

print()

analyst_auth = client.authorize(
    session_token=analyst_token.session_token,
    agent_id="analyst",
    action_type="LLM_CALL",
    description="Analyze findings",
    requested_quantity=50.00,
    claimed_category="llm_calls",
    idempotency_key="analyst-001",
)
print(f"Analyst: {analyst_auth.decision} — $50.00")

writer_auth = client.authorize(
    session_token=writer_token.session_token,
    agent_id="writer",
    action_type="LLM_CALL",
    description="Write report",
    requested_quantity=40.00,
    claimed_category="llm_calls",
    idempotency_key="writer-001",
)
print(f"Writer:  {writer_auth.decision} — $40.00")

print()
print("✓ Fleet completed. Rogue researcher capped at $200, rest of fleet unaffected.")

print(f"\n→ See the ledger: https://figuard-sandbox-1.onrender.com/ui")